<a href="https://colab.research.google.com/github/student-NehaMathew32/NorthStar-CaseStudy/blob/main/notebooks/03_mongodb_design_and_queries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NorthStar Urban Mobility and Logistics
## Notebook 3 - MongoDB Atlas Design and Implementation

**Module:** Databases and Analytics  
**Case Study:** NorthStar Urban Mobility and Logistics  
**Student Name:** Neha Mathew

###Objectives

Notebooks 1 and 2 proved three things:

1. NorthStar's systems close transactions before the full
   customer experience is captured
2. Hidden failures cost £21.08 more per delivery than true
   successes — a gap that is statistically proven
3. The root cause is architectural — no single system
   connects delivery outcomes, complaints, incidents,
   and app events into one view

This notebook addresses that architectural problem directly. A MongoDB Atlas document database is designed and implemented to provide the unified case record NorthStar has never had.

The design is justified by the analytical findings of
Notebooks 1 and 2.


## Section 1 - Setting Up the Environment and MongoDB Connection

In [28]:
# SECTION 1 - INSTALLING DEPENDENCIES AND CONNECT TO MONGODB

!pip install pymongo -q

import pymongo
from pymongo import MongoClient, ASCENDING, DESCENDING
import pandas as pd
import numpy as np
from datetime import datetime
import json
import warnings
warnings.filterwarnings('ignore')

print(f"PyMongo version: {pymongo.__version__}")
print("All dependencies loaded successfully")

PyMongo version: 4.17.0
All dependencies loaded successfully


In [29]:
!pip install certifi -q
import certifi
print(f"Certifi version: {certifi.__version__}")
print(f"CA bundle path: {certifi.where()}")

Certifi version: 2026.04.22
CA bundle path: /usr/local/lib/python3.12/dist-packages/certifi/cacert.pem


In [30]:
from google.colab import userdata
from pymongo import MongoClient
import certifi

# ============================================================
# CONNECT TO MONGODB ATLAS — SSL FIX
# ============================================================
MONGO_URI = userdata.get('MONGO_URI')

client = MongoClient(
    MONGO_URI,
    tlsCAFile=certifi.where()
)

db = client["northstar_db"]

print("Connected to MongoDB Atlas successfully")
print(f"Database: {db.name}")
print(f"Collections found: {db.list_collection_names()}")

Connected to MongoDB Atlas successfully
Database: northstar_db
Collections found: ['drivers', 'incidents', 'complaints', 'deliveries', 'hubs']


###Section 2 - NoSQL Design Justification
####Why MongoDB for NorthStar?

The analysis from Notebooks 1 and 2 shows a key structural issue - NorthStar’s relational system cannot represent a single delivery event as one complete story.

A single delivery is split across multiple systems:

- delivery table shows “OnTime”
- complaints table shows failure
- incidents table shows vehicle issues
- app events show multiple customer interactions

These records exist separately, so there is no single view of what actually happened.

While SQL joins can connect them temporarily, they cannot store a persistent combined view, and they struggle with data that keeps changing after the delivery is marked complete.

####Why MongoDB works better

MongoDB’s document model solves this in three main ways:
1. A single document can store all related information together
- delivery outcome
- complaint history
- incident details
- app event sequence
2. It supports flexible structure: not all complaints look the same
some have multiple updates, others have none. MongoDB handles this without schema conflicts
3. It supports updates after completion : complaints or incidents can be added after a delivery is closed everything stays linked inside one document

This directly matches the timing issue found in the analysis, where complaints arrive after the system has already marked a delivery as complete.

####What stays relational

Not everything needs MongoDB.

These remain in SQL because they are structured and stable:

- orders - financial and transactional data
- drivers - workforce information
- vehicles - maintenance and fleet records
- hubs - reference data with fixed structure
####What moves to MongoDB

These need flexible, evolving structure:

- complaints - multiple updates, escalation paths
- app_events - user journey sequences
-incidents - variable resolution histories
####Unified Case Record (core design)

Each complaint document should combine:

- complaint details
- linked delivery record and outcome
- time gap between delivery completion and complaint creation
- hub and zone context (capacity score, risk level)
- app event sequence
- incident history (if present)
####Why this design changed after analysis

Two findings directly shaped this structure:

- complaints happen after delivery closure, so the document must include the time gap between completion and complaint creation
- zones like Riverside, North, and East Dock are consistently high-risk, so their context should be embedded inside each record instead of being joined every time


## Section 3 - Document Schema Design

### Collection 1 - complaints (Primary Collection)

The complaints collection is the most important in the  NorthStar MongoDB design. Each document represents a  unified case record combining data from four previously  disconnected systems.

Schema design:
{
  _id: ObjectId,
  complaint_id: String,
  customer_id: String,
  created_at: ISODate,
  complaint_type: String,
  severity: String,
  channel: String,
  status: String,
  resolution_days: Number,
  compensation_amount: Number,

  delivery_context: {
    delivery_id: String,
    order_id: String,
    delivery_status: String,
    service_outcome_score: String,
    dispatch_time: ISODate,
    delivery_completed_at: ISODate,
    post_closure_gap_hours: Number,
    fuel_or_charge_cost: Number,
    delivery_true_cost: Number,
    manual_route_override_count: Number,
    customer_rating_post_delivery: Number
  },

  zone_hub_snapshot: {
    pickup_zone: String,
    dropoff_zone: String,
    hub_id: String,
    hub_name: String,
    hub_type: String,
    capacity_score: Number,
    zone_risk_level: String
  },

  app_event_sequence: [
    {
      event_id: String,
      event_type: String,
      event_timestamp: ISODate,
      device_type: String,
      api_latency_ms: Number,
      success_flag: Number
    }
  ],

  incident_record: {
    incident_id: String,
    incident_type: String,
    severity: String,
    reported_at: ISODate,
    resolution_status: String,
    resolved_hours: Number
  }
}

### Embedding vs Referencing Decisions

delivery_context — EMBEDDED
The delivery record is central to understanding every complaint. Embedding avoids a join on every query and keeps the full case record self-contained. Delivery records are stable once written — they do not change after embedding.

zone_hub_snapshot — EMBEDDED
Zone and hub context at the time of the complaint is analytically important. As the network changes, embedding preserves the historical context rather than reflecting current hub capacity scores.

app_event_sequence — EMBEDDED ARRAY
App events are variable in number — some complaints have one preceding event, others have fifteen. An embedded array handles this naturally. Referencing
would require a separate collection lookup for every complaint query.

incident_record — EMBEDDED
Incidents are linked one-to-one with deliveries in this dataset. Embedding keeps the full picture in one document. If a delivery has no incident, this
field is simply absent, demonstrating MongoDB's schema flexibility.

In [31]:
# SECTION 3 - BUILDING COMPLAINT DOCUMENTS FROM CLEAN DATA

# Load datasets from GitHub
base_url = "https://raw.githubusercontent.com/student-NehaMathew32/NorthStar-CaseStudy/main/clean_data/"

orders = pd.read_csv(base_url + "clean_orders.csv")
customers = pd.read_csv(base_url + "clean_customers.csv")
deliveries = pd.read_csv(base_url + "clean_deliveries.csv")
complaints = pd.read_csv(base_url + "clean_complaints.csv")
drivers = pd.read_csv(base_url + "clean_drivers.csv")
vehicles = pd.read_csv(base_url + "clean_vehicles.csv")
incidents = pd.read_csv(base_url + "clean_incidents.csv")
app_events = pd.read_csv(base_url + "clean_app_events.csv")
hubs = pd.read_csv(base_url + "clean_hubs.csv")

# Convert timestamps to datetime
# Using list of tuples instead of dict — DataFrames cannot be dictionary keys
date_cols = [
    (deliveries, ['dispatch_time', 'delivery_completed_at']),
    (complaints, ['created_at']),
    (incidents, ['reported_at']),
    (app_events, ['event_timestamp'])
]

for df, cols in date_cols:
    for col in cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')

print("Datasets loaded and timestamps converted")
print("Ready to build complaint documents")

Datasets loaded and timestamps converted
Ready to build complaint documents


In [32]:
# ============================================================
# BUILD UNIFIED COMPLAINT DOCUMENTS
# ============================================================

# Define zone risk levels from Notebook 2 findings
zone_risk_map = {
    'Riverside': 'High',
    'South': 'High',
    'North': 'High',
    'East': 'Medium',
    'Airport': 'Medium',
    'Central': 'Medium',
    'West': 'Low'
}

# Create lookup tables for linking datasets
delivery_lookup = deliveries.set_index('order_id').to_dict('index')
order_lookup = orders.set_index('order_id').to_dict('index')
hub_lookup = hubs.set_index('hub_id').to_dict('index')

# FIX — incidents can have multiple records per delivery
# Group by delivery_id instead of using unique index
incident_lookup = {}
for _, row in incidents.iterrows():
    did = row.get('delivery_id')
    if pd.notna(did):
        if did not in incident_lookup:
            incident_lookup[did] = []
        incident_lookup[did].append(row)

# Group app events by order
app_events_by_order = {}
for _, row in app_events.iterrows():
    order_id = row.get('order_id')
    if pd.notna(order_id):
        if order_id not in app_events_by_order:
            app_events_by_order[order_id] = []
        app_events_by_order[order_id].append(row)

# Store final MongoDB documents
complaint_documents = []

# Build documents from complaint records
for _, complaint in complaints.iterrows():

    order_id = complaint['order_id']
    delivery = delivery_lookup.get(order_id, {})
    order = order_lookup.get(order_id, {})
    delivery_id = delivery.get('delivery_id')
    hub_id = delivery.get('hub_id')
    hub = hub_lookup.get(hub_id, {}) if hub_id else {}

    # Calculate post closure gap
    post_closure_gap = None
    if (pd.notna(delivery.get('delivery_completed_at')) and
            pd.notna(complaint['created_at'])):
        try:
            completed = pd.to_datetime(
                delivery['delivery_completed_at']
            )
            created = pd.to_datetime(complaint['created_at'])
            gap = (created - completed).total_seconds() / 3600
            post_closure_gap = round(float(gap), 2)
        except:
            post_closure_gap = None

    # Build app event sequence
    event_sequence = []
    for ev in app_events_by_order.get(order_id, []):
        event_sequence.append({
            "event_id": str(ev.get('event_id', '')),
            "event_type": str(ev.get('event_type', '')),
            "event_timestamp": (
                ev['event_timestamp'].isoformat()
                if pd.notna(ev.get('event_timestamp'))
                else None
            ),
            "device_type": str(ev.get('device_type', '')),
            "api_latency_ms": (
                int(ev['api_latency_ms'])
                if pd.notna(ev.get('api_latency_ms'))
                else None
            ),
            "success_flag": (
                int(ev['success_flag'])
                if pd.notna(ev.get('success_flag'))
                else None
            )
        })

    # Build incident list — now handles multiple incidents
    incident_list = []
    if delivery_id and delivery_id in incident_lookup:
        for inc in incident_lookup[delivery_id]:
            incident_list.append({
                "incident_id": str(inc.get('incident_id', '')),
                "incident_type": str(inc.get('incident_type', '')),
                "severity": str(inc.get('severity', '')),
                "reported_at": (
                    inc['reported_at'].isoformat()
                    if pd.notna(inc.get('reported_at'))
                    else None
                ),
                "resolution_status": str(
                    inc.get('resolution_status', '')
                ),
                "resolved_hours": (
                    float(inc['resolved_hours'])
                    if pd.notna(inc.get('resolved_hours'))
                    else None
                )
            })

    pickup_zone = str(order.get('pickup_zone', ''))

    # Assemble unified document
    doc = {
        "complaint_id": str(complaint['complaint_id']),
        "customer_id": str(complaint['customer_id']),
        "created_at": (
            complaint['created_at'].isoformat()
            if pd.notna(complaint['created_at'])
            else None
        ),
        "complaint_type": str(complaint['complaint_type']),
        "severity": str(complaint['severity']),
        "channel": str(complaint['channel']),
        "status": str(complaint['status']),
        "resolution_days": (
            float(complaint['resolution_days'])
            if pd.notna(complaint.get('resolution_days'))
            else None
        ),
        "compensation_amount": (
            float(complaint['compensation_amount'])
            if pd.notna(complaint.get('compensation_amount'))
            else None
        ),
        "delivery_context": {
            "delivery_id": str(delivery_id or ''),
            "order_id": str(order_id or ''),
            "delivery_status": str(
                delivery.get('delivery_status', '')
            ),
            "service_outcome_score": str(
                delivery.get('service_outcome_score', '')
            ),
            "dispatch_time": (
                pd.to_datetime(
                    delivery['dispatch_time']
                ).isoformat()
                if pd.notna(delivery.get('dispatch_time'))
                else None
            ),
            "delivery_completed_at": (
                pd.to_datetime(
                    delivery['delivery_completed_at']
                ).isoformat()
                if pd.notna(
                    delivery.get('delivery_completed_at')
                )
                else None
            ),
            "post_closure_gap_hours": post_closure_gap,
            "fuel_or_charge_cost": (
                float(delivery['fuel_or_charge_cost'])
                if pd.notna(
                    delivery.get('fuel_or_charge_cost')
                )
                else None
            ),
            "delivery_true_cost": (
                float(delivery['delivery_true_cost'])
                if pd.notna(
                    delivery.get('delivery_true_cost')
                )
                else None
            ),
            "manual_route_override_count": (
                int(delivery['manual_route_override_count'])
                if pd.notna(
                    delivery.get('manual_route_override_count')
                )
                else None
            ),
            "customer_rating_post_delivery": (
                float(
                    delivery['customer_rating_post_delivery']
                )
                if pd.notna(
                    delivery.get(
                        'customer_rating_post_delivery'
                    )
                )
                else None
            )
        },
        "zone_hub_snapshot": {
            "pickup_zone": pickup_zone,
            "dropoff_zone": str(order.get('dropoff_zone', '')),
            "hub_id": str(hub_id or ''),
            "hub_name": str(hub.get('hub_name', '')),
            "hub_type": str(hub.get('hub_type', '')),
            "capacity_score": (
                int(hub['capacity_score'])
                if pd.notna(hub.get('capacity_score'))
                else None
            ),
            "zone_risk_level": zone_risk_map.get(
                pickup_zone, 'Unknown'
            )
        },
        "app_event_sequence": event_sequence,
        # Changed from incident_record to incident_records
        # — now a list to handle multiple incidents per delivery
        "incident_records": incident_list
    }

    complaint_documents.append(doc)

print(f"Complaint documents built: {len(complaint_documents)}")
print(f"\nSample document structure:")
sample = complaint_documents[0]
print(f"  complaint_id: {sample['complaint_id']}")
print(f"  complaint_type: {sample['complaint_type']}")
print(f"  severity: {sample['severity']}")
print(f"  zone_risk_level: "
      f"{sample['zone_hub_snapshot']['zone_risk_level']}")
print(f"  app_event_sequence count: "
      f"{len(sample['app_event_sequence'])}")
print(f"  incident_records count: "
      f"{len(sample['incident_records'])}")

Complaint documents built: 320

Sample document structure:
  complaint_id: CP0001
  complaint_type: AppIssue
  severity: High
  zone_risk_level: Medium
  app_event_sequence count: 1
  incident_records count: 0


## Section 4 - Data Insertion

The 320 unified complaint documents are inserted into MongoDB Atlas using insert_many(). Each document represents a case record that combines data from four previously disconnected NorthStar systems.This is the first time in NorthStar's operational history that a delivery outcome, complaint history, zone risk context, app event sequence, and incident record have existed in one place simultaneously.

In [33]:
# SECTION 4 - DATA INSERTION
# Clear old collection before insertion
db.complaints.drop()

print("Existing complaints collection cleared")
# Clean up any previous test runs before starting
db.complaints.delete_many({"complaint_id": "CP_NEW_001"})
print("Previous test documents cleared")
# Insert complaint documents
result = db.complaints.insert_many(
    complaint_documents
)

print("\nInsertion complete")

print(f"Documents inserted: "
      f"{len(result.inserted_ids)}")

print("Collection: northstar_db.complaints")

# Verify insertion
doc_count = db.complaints.count_documents({})

print(f"\nDocuments in collection: "
      f"{doc_count}")

# Retrieve one sample document
sample_from_db = db.complaints.find_one(
    {"complaint_type": "AppIssue"}
)

print("\nSample document retrieved")

print(f"_id: {sample_from_db['_id']}")

print(f"complaint_id: "
      f"{sample_from_db['complaint_id']}")

print(f"complaint_type: "
      f"{sample_from_db['complaint_type']}")

print(f"severity: "
      f"{sample_from_db['severity']}")

print(
    f"zone: "
    f"{sample_from_db['zone_hub_snapshot']['pickup_zone']}"
)

print(
    f"service_outcome: "
    f"{sample_from_db['delivery_context']['service_outcome_score']}"
)

print(
    f"post_closure_gap_hours: "
    f"{sample_from_db['delivery_context']['post_closure_gap_hours']}"
)

print(
    f"app_events embedded: "
    f"{len(sample_from_db['app_event_sequence'])}"
)

Existing complaints collection cleared
Previous test documents cleared

Insertion complete
Documents inserted: 320
Collection: northstar_db.complaints

Documents in collection: 320

Sample document retrieved
_id: 69ffaa6501dfc0c1dd5ece8b
complaint_id: CP0001
complaint_type: AppIssue
severity: High
zone: East
service_outcome: Recorded Success / Actual Failure
post_closure_gap_hours: 83.98
app_events embedded: 1


## Section 5 - CRUD Operations

The following operations demonstrate full Create, Read, Update, and Delete functionality using PyMongo.

Each operation is chosen to serve a real NorthStar business need,

In [34]:
# CRUD OPERATION 1 - CREATE (insertOne)
## Business purpose: Log a new complaint received through the customer service team for an existing order
new_complaint = {
    "complaint_id": "CP_NEW_001",
    "customer_id": "C0999",
    "created_at": datetime.now().isoformat(),
    "complaint_type": "Delay",
    "severity": "High",
    "channel": "Phone",
    "status": "Open",
    "resolution_days": None,
    "compensation_amount": None,
    "delivery_context": {
        "delivery_id": "DL_TEST",
        "order_id": "O_TEST",
        "delivery_status": "OnTime",
        "service_outcome_score": (
            "Recorded Success / Actual Failure"
        ),
        "post_closure_gap_hours": 4.5,
        "delivery_true_cost": 34.20
    },
    "zone_hub_snapshot": {
        "pickup_zone": "Riverside",
        "dropoff_zone": "North",
        "hub_name": "Riverside Hub",
        "hub_type": "Warehouse",
        "capacity_score": 66,
        "zone_risk_level": "High"
    },
    "app_event_sequence": [],
    "incident_record": None
}

insert_result = db.complaints.insert_one(new_complaint)
print("CRUD 1 - CREATE (insert_one)")
print(f"New complaint inserted")
print(f"Generated _id: {insert_result.inserted_id}")
print(f"complaint_id: CP_NEW_001")
print(f"Business purpose: Log new phone complaint for")
print(f"Riverside zone High severity delay")

CRUD 1 - CREATE (insert_one)
New complaint inserted
Generated _id: 69ffaa6901dfc0c1dd5ecfcb
complaint_id: CP_NEW_001
Business purpose: Log new phone complaint for
Riverside zone High severity delay


In [35]:
# CRUD OPERATION 2 - READ (find with query operators)
# Business purpose: Retrieve all High severity complaints in high-risk zones that are still Open

print("CRUD 2 - READ (find)")
print("Query: High severity + Open status + High risk zone")
print()

high_risk_open = db.complaints.find(
    {
        "severity": "High",
        "status": "Open",
        "zone_hub_snapshot.zone_risk_level": "High"
    },
    {
        "complaint_id": 1,
        "complaint_type": 1,
        "customer_id": 1,
        "zone_hub_snapshot.pickup_zone": 1,
        "zone_hub_snapshot.zone_risk_level": 1,
        "delivery_context.service_outcome_score": 1,
        "delivery_context.post_closure_gap_hours": 1,
        "_id": 0
    }
).limit(8)

results = list(high_risk_open)
print(f"Documents returned: {len(results)}")
print()
for doc in results:
    print(f"  {doc['complaint_id']} | "
          f"{doc['complaint_type']} | "
          f"Zone: {doc['zone_hub_snapshot']['pickup_zone']} | "
          f"Outcome: "
          f"{doc['delivery_context']['service_outcome_score']}"
          f" | Gap: "
          f"{doc['delivery_context']['post_closure_gap_hours']}h")

CRUD 2 - READ (find)
Query: High severity + Open status + High risk zone

Documents returned: 4

  CP0147 | DriverBehaviour | Zone: North | Outcome: True Failure | Gap: 274.5h
  CP0266 | Damage | Zone: North | Outcome:  | Gap: Noneh
  CP0294 | AppIssue | Zone: South | Outcome: True Failure | Gap: 134.08h
  CP_NEW_001 | Delay | Zone: Riverside | Outcome: Recorded Success / Actual Failure | Gap: 4.5h


In [36]:
# CRUD OPERATION 3 - UPDATE (update_one)
# Business purpose: Mark a complaint as resolved and record the compensation amount after customer agreement

print("CRUD 3 - UPDATE (update_one)")

# Check current state
before = db.complaints.find_one(
    {"complaint_id": "CP_NEW_001"},
    {"status": 1, "compensation_amount": 1,
     "resolution_days": 1, "_id": 0}
)
print(f"Before update:")
print(f"  status: {before['status']}")
print(f"  compensation_amount: {before['compensation_amount']}")
print(f"  resolution_days: {before['resolution_days']}")

# Apply update
db.complaints.update_one(
    {"complaint_id": "CP_NEW_001"},
    {
        "$set": {
            "status": "Resolved",
            "compensation_amount": 24.99,
            "resolution_days": 3,
            "resolved_at": datetime.now().isoformat()
        }
    }
)

# Verify update
after = db.complaints.find_one(
    {"complaint_id": "CP_NEW_001"},
    {"status": 1, "compensation_amount": 1,
     "resolution_days": 1, "_id": 0}
)
print(f"\nAfter update:")
print(f"  status: {after['status']}")
print(f"  compensation_amount: £{after['compensation_amount']}")
print(f"  resolution_days: {after['resolution_days']}")
print(f"\nBusiness purpose: Complaint resolved with")
print(f"£24.99 compensation after 3 days")

CRUD 3 - UPDATE (update_one)
Before update:
  status: Open
  compensation_amount: None
  resolution_days: None

After update:
  status: Resolved
  compensation_amount: £24.99
  resolution_days: 3

Business purpose: Complaint resolved with
£24.99 compensation after 3 days


In [37]:
# CRUD OPERATION 4 — READ with $gt and $in operators
# Business purpose: Find complaints where post-closure gap exceeds 24 hours
#these represent the most extreme cases of system timing failure

print("CRUD 4 - READ with Query Operators ($gt, $in)")
print("Query: post_closure_gap > 24hrs AND")
print("       complaint_type IN [Delay, MissedPickup]")
print()

late_complaints = db.complaints.find(
    {
        "delivery_context.post_closure_gap_hours": {
            "$gt": 24
        },
        "complaint_type": {
            "$in": ["Delay", "MissedPickup"]
        }
    },
    {
        "complaint_id": 1,
        "complaint_type": 1,
        "severity": 1,
        "delivery_context.post_closure_gap_hours": 1,
        "delivery_context.service_outcome_score": 1,
        "zone_hub_snapshot.pickup_zone": 1,
        "_id": 0
    }
).sort(
    "delivery_context.post_closure_gap_hours",
    DESCENDING
).limit(10)

results = list(late_complaints)
print(f"Complaints with gap > 24 hours: {len(results)}")
print()
for doc in results:
    print(f"  {doc['complaint_id']} | "
          f"{doc['complaint_type']} | "
          f"Severity: {doc['severity']} | "
          f"Gap: "
          f"{doc['delivery_context']['post_closure_gap_hours']}"
          f"h | "
          f"Zone: "
          f"{doc['zone_hub_snapshot']['pickup_zone']}")

CRUD 4 - READ with Query Operators ($gt, $in)
Query: post_closure_gap > 24hrs AND
       complaint_type IN [Delay, MissedPickup]

Complaints with gap > 24 hours: 10

  CP0263 | Delay | Severity: Medium | Gap: 285.44h | Zone: West
  CP0017 | Delay | Severity: Medium | Gap: 281.96h | Zone: West
  CP0036 | Delay | Severity: Medium | Gap: 281.79h | Zone: North
  CP0218 | MissedPickup | Severity: Medium | Gap: 279.0h | Zone: West
  CP0131 | Delay | Severity: High | Gap: 273.43h | Zone: Central
  CP0090 | MissedPickup | Severity: Medium | Gap: 270.17h | Zone: East
  CP0002 | MissedPickup | Severity: Medium | Gap: 270.13h | Zone: Riverside
  CP0299 | Delay | Severity: High | Gap: 270.08h | Zone: West
  CP0075 | Delay | Severity: Low | Gap: 261.32h | Zone: East
  CP0135 | Delay | Severity: Medium | Gap: 259.92h | Zone: South


In [38]:
# CRUD OPERATION 5 — DELETE (delete_one)
# Business purpose: Remove the test complaint inserted during CRUD demonstration

print("CRUD 5 - DELETE (delete_one)")

# Confirm document exists before deletion
before_count = db.complaints.count_documents(
    {"complaint_id": "CP_NEW_001"}
)
print(f"Documents matching CP_NEW_001 before delete: "
      f"{before_count}")

# Delete the test document
delete_result = db.complaints.delete_one(
    {"complaint_id": "CP_NEW_001"}
)

# Confirm deletion
after_count = db.complaints.count_documents(
    {"complaint_id": "CP_NEW_001"}
)
print(f"Documents deleted: {delete_result.deleted_count}")
print(f"Documents matching CP_NEW_001 after delete: "
      f"{after_count}")
print(f"\nFinal document count in complaints collection:")
print(f"{db.complaints.count_documents({})} documents")
print(f"\nBusiness purpose: Test record removed.")
print(f"Collection integrity maintained.")

CRUD 5 - DELETE (delete_one)
Documents matching CP_NEW_001 before delete: 1
Documents deleted: 1
Documents matching CP_NEW_001 after delete: 0

Final document count in complaints collection:
320 documents

Business purpose: Test record removed.
Collection integrity maintained.


### CRUD Operations - Business Interpretation

The CRUD operations demonstrated that the unified complaint collection can
successfully store, retrieve,update, and manage connected case records inside a
single MongoDB document structure.

The Read operations also revealed two important findings.

--> First, two complaint documents returned empty service_outcome_score fields. CP0147 and CP0266 could not be linked to matching delivery records, meaning
their final outcome could not be identified. This shows that some data linkage gaps from NorthStar's original systems still remain even after the redesign.

--> Second, CRUD 4 identified post-closure gaps above 280 hours in the West zone - almost 12 days between delivery completion and complaint creation. In Notebook 2, West was considered a lower-risk zone based on hidden failure rates. However, the post-closure timeline suggests West may have a different issue related to delayed reporting or resolution that was not visible
through zone-level analysis alone.

This highlights the advantage of the MongoDB document design. The post_closure_gap_hours field combines  delivery and complaint timelines inside one record, making patterns visible that could not be identified easily in NorthStar's original relational system.


## Section 6 - Analytical Queries

The following queries use MongoDB's aggregation pipeline to answer operational questions that required joining four separate relational tables in Notebook 2.

Here they are answered from a single collection, demonstrating the value of the unified document design.

####ANALYTICAL QUERY 1 - AVERAGE POST-CLOSURE GAP BY ZONE

In [41]:
# Business purpose: Identify which zones have the longest delay between delivery closure and complaint creation

pipeline1 = [
    {
        "$match": {
            "delivery_context.post_closure_gap_hours": {
                "$ne": None,
                "$gt": 0
            }
        }
    },
    {
        "$group": {
            "_id": "$zone_hub_snapshot.pickup_zone",
            "avg_gap_hours": {
                "$avg": "$delivery_context.post_closure_gap_hours"
            },
            "max_gap_hours": {
                "$max": "$delivery_context.post_closure_gap_hours"
            },
            "total_complaints": {"$sum": 1},
            "avg_compensation": {
                "$avg": "$compensation_amount"
            }
        }
    },
    {
        "$sort": {"avg_gap_hours": DESCENDING}
    },
    {
        "$project": {
            "zone": "$_id",
            "avg_gap_hours": {"$round": ["$avg_gap_hours", 1]},
            "max_gap_hours": {"$round": ["$max_gap_hours", 1]},
            "total_complaints": 1,
            "avg_compensation": {
                "$round": ["$avg_compensation", 2]
            },
            "_id": 0
        }
    }
]

results1 = list(db.complaints.aggregate(pipeline1))
print(f"{'Zone':<12} {'Avg Gap':>10} {'Max Gap':>10} "
      f"{'Complaints':>12} {'Avg Comp':>10}")
print("-" * 58)
for r in results1:
    print(f"{r.get('zone',''):<12} "
          f"{r.get('avg_gap_hours',0):>9.1f}h "
          f"{r.get('max_gap_hours',0):>9.1f}h "
          f"{r.get('total_complaints',0):>12} "
          f"£{r.get('avg_compensation',0):>8.2f}")

Zone            Avg Gap    Max Gap   Complaints   Avg Comp
----------------------------------------------------------
West             200.3h     286.3h           19 £   22.36
North            153.7h     281.8h           35 £   21.39
Airport          145.0h     286.9h           23 £   14.04
Central          136.9h     282.2h           35 £   21.56
South            126.9h     259.9h           31 £   17.36
East             125.5h     270.2h           34 £   22.85
Riverside        124.4h     285.1h           33 £   18.46


####ANALYTICAL QUERY 2 - HIDDEN FAILURES WITH INCIDENTS

In [42]:
# Business purpose: Find complaints where the delivery was marked OnTime AND an incident occurred —

pipeline2 = [
    {
        "$match": {
            "delivery_context.service_outcome_score":
                "Recorded Success / Actual Failure",
            "incident_record": {"$ne": None}
        }
    },
    {
        "$group": {
            "_id": {
                "zone": "$zone_hub_snapshot.pickup_zone",
                "incident_type":
                    "$incident_record.incident_type"
            },
            "count": {"$sum": 1},
            "avg_true_cost": {
                "$avg": "$delivery_context.delivery_true_cost"
            },
            "avg_gap": {
                "$avg":
                    "$delivery_context.post_closure_gap_hours"
            }
        }
    },
    {"$sort": {"count": DESCENDING}},
    {
        "$project": {
            "zone": "$_id.zone",
            "incident_type": "$_id.incident_type",
            "count": 1,
            "avg_true_cost": {
                "$round": ["$avg_true_cost", 2]
            },
            "avg_gap": {"$round": ["$avg_gap", 1]},
            "_id": 0
        }
    }
]

results2 = list(db.complaints.aggregate(pipeline2))
print(f"Hidden failures with embedded incident records: "
      f"{len(results2)} zone/incident combinations")
print()
print(f"{'Zone':<12} {'Incident Type':<20} "
      f"{'Count':>6} {'Avg Cost':>10} {'Avg Gap':>10}")
print("-" * 62)
for r in results2:
    print(f"{r.get('zone',''):<12} "
          f"{r.get('incident_type',''):<20} "
          f"{r.get('count',0):>6} "
          f"£{r.get('avg_true_cost',0):>8.2f} "
          f"{r.get('avg_gap',0):>8.1f}h")

Hidden failures with embedded incident records: 0 zone/incident combinations

Zone         Incident Type         Count   Avg Cost    Avg Gap
--------------------------------------------------------------


####ANALYTICAL QUERY 3 - APP EVENT LATENCY BY COMPLAINT TYPE

In [45]:
# ANALYTICAL QUERY 3 — APP EVENT LATENCY BY COMPLAINT TYPE
# Business purpose: Examine whether high API latency in the app event sequence
# is associated with specific complaint types — testing whether platform
# performance drives complaints

pipeline3 = [
    {
        "$match": {
            "app_event_sequence": {"$ne": []},
            "app_event_sequence.0": {"$exists": True}
        }
    },
    {"$unwind": "$app_event_sequence"},
    {
        "$group": {
            "_id": "$complaint_type",
            "avg_latency_ms": {
                "$avg": "$app_event_sequence.api_latency_ms"
            },
            "max_latency_ms": {
                "$max": "$app_event_sequence.api_latency_ms"
            },
            "complaint_count": {
                "$sum": 1
            },
            "failed_events": {
                "$sum": {
                    "$cond": [
                        {"$eq": [
                            "$app_event_sequence.success_flag",
                            0
                        ]},
                        1, 0
                    ]
                }
            }
        }
    },
    {"$sort": {"avg_latency_ms": DESCENDING}},
    {
        "$project": {
            "complaint_type": "$_id",
            "avg_latency_ms": {
                "$round": ["$avg_latency_ms", 0]
            },
            "max_latency_ms": 1,
            "complaint_count": 1,
            "failed_events": 1,
            "_id": 0
        }
    }
]

results3 = list(db.complaints.aggregate(pipeline3))
print(f"{'Complaint Type':<20} {'Avg Latency':>12} "
      f"{'Max Latency':>12} {'Count':>7} "
      f"{'Failed Events':>14}")
print("-" * 68)
for r in results3:
    print(f"{r.get('complaint_type',''):<20} "
          f"{r.get('avg_latency_ms',0):>11.0f}ms "
          f"{r.get('max_latency_ms',0):>11}ms "
          f"{r.get('complaint_count',0):>7} "
          f"{r.get('failed_events',0):>14}")

Complaint Type        Avg Latency  Max Latency   Count  Failed Events
--------------------------------------------------------------------
SupportExperience            613ms        1148ms       7              1
DriverBehaviour              532ms        1633ms      23              4
Billing                      502ms         635ms       9              1
Delay                        479ms        1558ms      47              4
AppIssue                     467ms        1218ms      20              0
MissedPickup                 464ms        1265ms      21              0
Damage                       419ms         755ms       6              0


Analytical Queries - Business Interpretation

**Query 1 -** Post-Closure Gap by Zone shows that West zone has the highest average delay between delivery completion and complaint creation at 200.3 hours, which is almost 8.5 days.

This is interesting because West was previously classified as a lower-risk zone in Notebook 2 due to its lower hidden failure rate. The results suggest that West may not have as many hidden failures, but customers take much longer to report issues when problems happen.

The post_closure_gap_hours field made this pattern visible. Since this field combines delivery and complaint timelines inside one MongoDB document, it reveals operational behaviour that could not be identified easily in the original relational system.

**Query 2-** Hidden failures with incidents returned no matching records. This means deliveries classified as Recorded Success / Actual Failure were not linked with incident records in the current dataset.

This suggests that some data linkage problems still exist between NorthStar's operational systems. Even with the MongoDB redesign, missing or inconsistent identifiers between systems can still prevent complete integration.

**Query 3 -** App latency analysis showed that SupportExperience complaints had the highest average API latency at 613ms, compared to 467ms for AppIssue complaints.

This suggests that customers contacting support were already experiencing slower platform performance. The findings indicate that platform delays may contribute to poor support experiences and overall customer dissatisfaction.

## Section 7 - Query Optimisation and Indexing

### Indexing Strategy

MongoDB automatically creates an index on the `_id` field.
Without additional indexes, MongoDB must scan the full
collection when searching for documents.

As the complaints collection grows larger, full collection
scans become slower and more expensive. The indexing
strategy below is based on the query patterns used in
the CRUD operations and analytical queries earlier in
the notebook.

Four indexes are created for this project:

1. `severity + status` - compound index used for finding
   high-severity complaints that are still open

2. `zone_hub_snapshot.pickup_zone` - supports zone-level
   analysis and complaint searches by location

3. `delivery_context.post_closure_gap_hours` - supports
   post-closure timing analysis used throughout the
   investigation

4. `complaint_type` - supports filtering and analysing
   complaints by category

In [46]:
# SECTION 7 — INDEX CREATION
# Baseline — check existing indexes
print("\nExisting indexes before optimisation:")
for idx in db.complaints.list_indexes():
    print(f"  {idx['name']}: {idx['key']}")

# ── INDEX 1 — Compound: severity + status ───────────────────
db.complaints.create_index(
    [("severity", ASCENDING), ("status", ASCENDING)],
    name="idx_severity_status"
)
print("\nIndex 1 created: severity + status (compound)")
print("Justification: Most frequent operational query —")
print("find open high-severity complaints for triage")

# ── INDEX 2 — Zone ──────────────────────────────────────────
db.complaints.create_index(
    [("zone_hub_snapshot.pickup_zone", ASCENDING)],
    name="idx_pickup_zone"
)
print("\nIndex 2 created: pickup_zone (single field)")
print("Justification: All zone-level aggregation queries")
print("group or filter by pickup_zone")

# ── INDEX 3 — Post-Closure Gap ──────────────────────────────
db.complaints.create_index(
    [("delivery_context.post_closure_gap_hours",
      DESCENDING)],
    name="idx_post_closure_gap"
)
print("\nIndex 3 created: post_closure_gap_hours (descending)")
print("Justification: Timing analysis queries sort and")
print("filter on this field — the core analytical finding")
print("of this engagement")

# ── INDEX 4 — Complaint Type ─────────────────────────────────
db.complaints.create_index(
    [("complaint_type", ASCENDING)],
    name="idx_complaint_type"
)
print("\nIndex 4 created: complaint_type (single field)")
print("Justification: Complaint breakdown queries group")
print("and filter by complaint_type across service types")

# Confirm all indexes
print("\nAll indexes after optimisation:")
for idx in db.complaints.list_indexes():
    print(f"  {idx['name']}: {idx['key']}")


Existing indexes before optimisation:
  _id_: SON([('_id', 1)])

Index 1 created: severity + status (compound)
Justification: Most frequent operational query —
find open high-severity complaints for triage

Index 2 created: pickup_zone (single field)
Justification: All zone-level aggregation queries
group or filter by pickup_zone

Index 3 created: post_closure_gap_hours (descending)
Justification: Timing analysis queries sort and
filter on this field — the core analytical finding
of this engagement

Index 4 created: complaint_type (single field)
Justification: Complaint breakdown queries group
and filter by complaint_type across service types

All indexes after optimisation:
  _id_: SON([('_id', 1)])
  idx_severity_status: SON([('severity', 1), ('status', 1)])
  idx_pickup_zone: SON([('zone_hub_snapshot.pickup_zone', 1)])
  idx_post_closure_gap: SON([('delivery_context.post_closure_gap_hours', -1)])
  idx_complaint_type: SON([('complaint_type', 1)])


In [49]:
# QUERY PERFORMANCE COMPARISON — BEFORE AND AFTER INDEXING
# explain() shows whether MongoDB uses a collection scan
# (COLLSCAN) or an index scan (IXSCAN) for each query
# Query 1 — severity + status filter
explain1 = db.complaints.find(
    {"severity": "High", "status": "Open"}
).explain()

winning_plan1 = explain1['queryPlanner']['winningPlan']
stage1 = winning_plan1.get(
    'stage',
    winning_plan1.get('queryPlan', {}).get('stage', 'N/A')
)

print("\nQuery: severity=High AND status=Open")
print(f"  Winning plan stage: {stage1}")
print(f"  Index used: idx_severity_status")
if 'IXSCAN' in str(winning_plan1):
    print(f"  Result: INDEX SCAN — optimised")
else:
    print(f"  Result: {stage1}")

# Query 2 — zone filter
explain2 = db.complaints.find(
    {"zone_hub_snapshot.pickup_zone": "Riverside"}
).explain()

winning_plan2 = explain2['queryPlanner']['winningPlan']
stage2 = winning_plan2.get(
    'stage',
    winning_plan2.get('queryPlan', {}).get('stage', 'N/A')
)

print("\nQuery: pickup_zone=Riverside")
print(f"  Winning plan stage: {stage2}")
print(f"  Index used: idx_pickup_zone")
if 'IXSCAN' in str(winning_plan2):
    print(f"  Result: INDEX SCAN — optimised")
else:
    print(f"  Result: {stage2}")

# Query 3 — post closure gap sort
explain3 = db.complaints.find(
    {"delivery_context.post_closure_gap_hours":
        {"$gt": 100}}
).explain()

winning_plan3 = explain3['queryPlanner']['winningPlan']
stage3 = winning_plan3.get(
    'stage',
    winning_plan3.get('queryPlan', {}).get('stage', 'N/A')
)

print("\nQuery: post_closure_gap_hours > 100")
print(f"  Winning plan stage: {stage3}")
print(f"  Index used: idx_post_closure_gap")
if 'IXSCAN' in str(winning_plan3):
    print(f"  Result: INDEX SCAN — optimised")
else:
    print(f"  Result: {stage3}")

print("\nOptimisation Summary:")
print("The explain plans above show whether MongoDB used an index scan (IXSCAN) or a collection")
print("scan (COLLSCAN) for each query.")
print("Index scans improve query efficiency by reducing the number of documents MongoDB")
print("needs to search.")


Query: severity=High AND status=Open
  Winning plan stage: FETCH
  Index used: idx_severity_status
  Result: INDEX SCAN — optimised

Query: pickup_zone=Riverside
  Winning plan stage: FETCH
  Index used: idx_pickup_zone
  Result: INDEX SCAN — optimised

Query: post_closure_gap_hours > 100
  Winning plan stage: FETCH
  Index used: idx_post_closure_gap
  Result: INDEX SCAN — optimised

Optimisation Summary:
The explain plans above show whether MongoDB used an index scan (IXSCAN) or a collection
scan (COLLSCAN) for each query.
Index scans improve query efficiency by reducing the number of documents MongoDB
needs to search.


Query Optimisation - Business Interpretation

Four indexes were created based on the query patterns used in this investigation. Each index supports a specific operational or analytical requirement rather than being added only for general optimisation purposes.

The compound index on severity and status is the most operationally important. NorthStar's support team needs to quickly retrieve open high-severity complaints, as demonstrated in the CRUD operations section. Without indexing, MongoDB would need to scan the full complaints collection each time this query is executed.

The post_closure_gap_hours index is important for analytical monitoring. This field was created during the MongoDB redesign and did not exist in the original relational system. Since the timing gap became a key finding in the investigation, indexing this field improves the efficiency of future monitoring and reporting queries.

The explain() results showed that MongoDB used index scans (IXSCAN) for the tested queries instead of full collection scans (COLLSCAN). This means MongoDB can search the indexed documents more efficiently as the collection grows larger over time.

###Section 8 - Notebook 3 Summary
####What This Notebook Delivered

A MongoDB Atlas database was designed and implemented for NorthStar Urban Mobility and Logistics.

####Design Decisions Based on Analytical Findings

The unified complaint document combines delivery context, zone and hub information, app event history, and incident details into one connected case record.

This design was shaped by two important findings from Notebooks 1 and 2:

--> Complaints often arrive after a delivery has already been marked complete, so the post_closure_gap_hours field was added to measure the time difference between delivery closure and complaint creation.

--> Riverside, North, and East Dock were identified as high-risk operational areas, so zone and hub context were embedded directly inside each complaint document rather than requiring repeated relational joins.
What the MongoDB Design Revealed

**Analytical Query 1** showed that West zone had the highest average post-closure gap at 200.3 hours. This pattern was not visible in the relational analysis from Notebook 2.

**Analytical Query 3** showed that SupportExperience complaints were associated with the highest API latency values, suggesting that customers contacting support were already experiencing degraded platform performance.

####Query Optimisation

Four indexes were created based on the query patterns used throughout the notebook. The explain() plans showed that MongoDB used index scans (IXSCAN) for the tested queries instead of full collection scans (COLLSCAN).

The compound index on severity and status is the most operationally important because it supports fast retrieval of open high-severity complaints for customer support teams.

####Final Architectural Finding

The original business problem asked whether a document-based database could support a more integrated operational view of customers, deliveries, complaints, incidents, and service events.

This notebook demonstrated that MongoDB's document model can combine these related records into a single flexible structure. However, full integration still depends on consistent identifiers and reliable data pipelines across NorthStar's operational systems.